# MNIST digit classification

Runs end to end on the SageMaker notebook instance:

1. download MNIST
2. upload it to the project S3 bucket (KMS encrypted)
3. train a classifier locally on this instance
4. write the model artifact back to S3

Training runs on the notebook instance itself, not a separate SageMaker
training job, so it stays inside the `ml.t3.medium` you are already paying for.

## 1. Setup

The bucket name comes from the Terraform output `data_bucket`. The execution
role is scoped to this bucket only, so pointing elsewhere will fail with
`AccessDenied`.

`conda_python3` is a minimal kernel, so scikit-learn is installed first. If you
would rather not install anything, switch the kernel to one of the preinstalled
ML images (Kernel > Change kernel) and skip the next cell.

In [1]:
%pip install -q scikit-learn

Note: you may need to restart the kernel to use updated packages.
scikit-learn 1.7.2


In [2]:
import boto3

REGION = "ca-central-1"

# From `terraform -chdir=infra/mlops output data_bucket`. The suffix is random,
# so it changes on every destroy/apply cycle.
BUCKET = "sagemaker-notebook-dev-data-3opmm6"

RAW_KEY = "raw/mnist/mnist.npz"
MODEL_KEY = "models/mnist/model.joblib"

s3 = boto3.client("s3", region_name=REGION)

# Fails loudly now rather than midway through training.
s3.head_bucket(Bucket=BUCKET)
print(f"bucket reachable: {BUCKET}")

ClientError: An error occurred (404) when calling the HeadBucket operation: Not Found

## 2. Download MNIST

`fetch_openml` pulls ~11 MB. This is the step that proves the subnet has a
route to the internet — if the notebook were on an isolated subnet it would
hang here.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")

X = mnist.data.astype(np.float32) / 255.0
y = mnist.target.astype(np.int64)

print(f"X: {X.shape}  y: {y.shape}")

## 3. Upload to S3

Written under `raw/`, matching the bucket layout. The bucket applies `aws:kms`
by default, so this also exercises `kms:GenerateDataKey` on the project CMK.

In [ ]:
import io

buf = io.BytesIO()
np.savez_compressed(buf, X=X, y=y)
buf.seek(0)

s3.upload_fileobj(buf, BUCKET, RAW_KEY)

size_mb = s3.head_object(Bucket=BUCKET, Key=RAW_KEY)["ContentLength"] / 1024**2
print(f"s3://{BUCKET}/{RAW_KEY}  ({size_mb:.1f} MB)")

## 4. Train

Logistic regression on a 10k subset. `ml.t3.medium` has 2 vCPU, so the full
70k rows would take a while — the subset keeps this to roughly a minute while
still landing around 90% accuracy.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

N = 10_000
X_train, X_test, y_train, y_test = train_test_split(
    X[:N], y[:N], test_size=0.2, random_state=42, stratify=y[:N]
)

# n_jobs is deprecated in sklearn >= 1.8 and has no effect on the default solver.
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

acc = accuracy_score(y_test, clf.predict(X_test))
print(f"test accuracy: {acc:.4f}")

## 5. Save the model to S3

In [ ]:
import joblib

buf = io.BytesIO()
joblib.dump(clf, buf)
buf.seek(0)

s3.upload_fileobj(buf, BUCKET, MODEL_KEY)
print(f"s3://{BUCKET}/{MODEL_KEY}")

## 6. Verify the round trip

Reload the model straight from S3 and predict, confirming both read and
KMS decrypt work.

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key=MODEL_KEY)
reloaded = joblib.load(io.BytesIO(obj["Body"].read()))

print(f"reloaded accuracy: {accuracy_score(y_test, reloaded.predict(X_test)):.4f}")

for o in s3.list_objects_v2(Bucket=BUCKET).get("Contents", []):
    if o["Size"]:
        print(f"  {o['Key']}  {o['Size'] / 1024**2:.1f} MB")